In [1]:
install.packages("table1")

Installing package into ‘/home/jupyter/.R/library’
(as ‘lib’ is unspecified)



In [2]:
library(bigrquery)
library(knitr)
library(tidyverse)
library(ggplot2)
library("table1")
library("IRdisplay")
bq_auth() 

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.2     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.0
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.2     ✔ tidyr     1.3.0
✔ purrr     1.0.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘table1’


The following objects are masked from ‘package:base’:

    units, units<-




In [3]:
data <-bq_dataset_query(query="
SELECT *

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_GLDIMD`"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

GLD <- bq_table_download(data) 

In [4]:
data1 <-bq_dataset_query(query="
select round(avg(count1),5)

from(

SELECT person_id,count(person_id) count1

 FROM `yhcr-prd-phm-bia-core.CB_1935_AK.src_bmbc_EYFSP_GLD`

 group by person_id)base"
                       ,x="yhcr-prd-phm-bia-core.CB_1935_AK")

GLDAVG <- bq_table_download(data1) 

In [5]:
display_jupyter <- function(x) {
  css <- system.file("table1_defaults_1.0/table1_defaults.css", package="table1")
  css <- paste(readLines(css), collapse="\n")
  x <- htmltools::tagList(htmltools::tags$style(css), htmltools::tags$div(class="Rtable1", x))
  IRdisplay::display_html(as.character(x))
}

In [6]:
GLD$Gender[(GLD$Gender) == "F"] <- "Female"
GLD$Gender[(GLD$Gender) == "M"] <- "Male"
GLD$Gender[(GLD$Gender) == "U"] <- "Unknown"

In [9]:
label(GLD$AcademicYear)      <- "Academic Year"
label(GLD$IMD)      <- "Average IMD"

label(GLD$FSMEligible)      <- "FSM Eligiblity"
label(GLD$ethnic_group)      <- "Ethnic Group"
label(GLD$Gender)      <- "Gender"


In [10]:
 x<-table1(~ GLD$AcademicYear + GLD$FSMEligible+GLD$Gender + GLD$ethnic_group+GLD$IMD|GLD$score,data=GLD
          ,justify=c("left", "left", rep("center", 5)),format_number = TRUE
          ,caption = "Table presenting GLD - Good Learning Development results from the Early years foundation stage profile (EYFSP)  "
    
         )

In [11]:
display_jupyter(x)

,1. True(N=45741),2. False(N=30037),Overall(N=75778)
Academic Year,,,
2012/2013,5268 (11.5%),6063 (20.2%),11331 (15.0%)
2013/2014,6118 (13.4%),5242 (17.5%),11360 (15.0%)
2014/2015,6749 (14.8%),4392 (14.6%),11141 (14.7%)
2015/2016,7132 (15.6%),3797 (12.6%),10929 (14.4%)
2016/2017,7050 (15.4%),3703 (12.3%),10753 (14.2%)
2017/2018,6781 (14.8%),3585 (11.9%),10366 (13.7%)
2018/2019,6643 (14.5%),3255 (10.8%),9898 (13.1%)
FSM Eligiblity,,,
Yes,7021 (15.3%),7582 (25.2%),14603 (19.3%)


In [12]:
GLD

person_id,ethnic_group,Ethnicity,Gender,BirthYear,AcademicYear,score,Metric,FSMEligible,IMD
<int>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<lgl>,<chr>
860171,White,Irish,Male,2008,2012/2013,2. False,GLD,FALSE,No Data
12529327,White,Irish,Female,2008,2012/2013,1. True,GLD,FALSE,5
12913837,White,Irish,Female,2008,2012/2013,2. False,GLD,FALSE,4
432907,White,Irish,Male,2008,2012/2013,2. False,GLD,FALSE,No Data
13322830,White,Irish,Female,2012,2016/2017,1. True,GLD,TRUE,No Data
13322830,White,Irish,Female,2012,2016/2017,1. True,GLD,FALSE,No Data
13421029,White,Irish,Male,2012,2016/2017,2. False,GLD,FALSE,No Data
12517337,White,Irish,Female,2008,2012/2013,2. False,GLD,TRUE,No Data
12496471,White,Irish,Male,2008,2012/2013,2. False,GLD,TRUE,5


In [28]:
OverallTrend1 <-GLD %>% select(1,6,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score)


PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "1. True" ))
PS1 <- PS1 %>% select(1,4)

PS1<- group_by(PS1, AcademicYear) %>% mutate(PctPerYear =(sum(percent)))
PS1 <- PS1 %>% select(1,3)

PS1 <- distinct(PS1)


OVERALL<-ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = 1)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication",atop("GLD - Achieving Good Learning Development Early years foundation stage profile (EYFSP)"), 
                           atop(italic("percentage of people achieveing atleast 'Expected' grade"), "")))) +


labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 90, vjust = 0.5, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.5,  vjust = -1) +
                   #+
  theme_minimal()

In [29]:
OverallTrend1 <- GLD %>% select(1,2,6,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,ethnic_group) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,ethnic_group)


PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)


#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "1. True" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)

ethnicity <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = ethnic_group,color = ethnic_group)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing Good Learning Development"),
                                                                                  atop(italic("By ethnic group")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -2) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [30]:
OverallTrend1 <- GLD %>% select(1,4,6,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,Gender) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,Gender)


PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "1. True" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,Gender) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)
PS1 <-PS1<-unique(filter(PS1,Gender == "Female" |Gender == "Male" ))

gender <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = Gender,color = Gender)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing Good Learning Development"),
                                                                                  atop(italic("By Gender")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [ ]:
OverallTrend1 <- GLD %>% select(1,10,6,7)

PersonalSocial1 <- OverallTrend1 %>%
  group_by(AcademicYear,score,IMD) 

PS1<- PersonalSocial1 %>% count(AcademicYear,score,IMD)


PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(percent = (n/sum(n))*100)
#PS1<-unique(filter(PS1,result == ))

#PS1<- group_by(PS1, AcademicYear,ethnic_group) %>% mutate(percent = (n/sum(n))*100)

PS1<-unique(filter(PS1,score == "1. True" ))
PS1 <- PS1 %>% select(1,3,5)

PS1<- group_by(PS1, AcademicYear,IMD) %>% mutate(PctPerYear =round(sum(percent)),2)
PS1 <- PS1 %>% select(1,2,4)

PS1 <- distinct(PS1)

 IMD <- ggplot(PS1 , aes(x = AcademicYear, y = PctPerYear,group = IMD,color = IMD)) +
#scale_y_continuous( limits=c(0,1))+
  geom_line() +
  geom_point() +
   ggtitle(expression(atop("COS-EY 1: 1.2 Speech, Language & Communication", atop(italic("percentage of people achieveing Good Learning Development"),
                                                                                  atop(italic("By IMD")))))) +
  labs(
       x = "Academic Year",
       y = "Percentage of students per year") +
scale_y_continuous(labels = function(x) paste0(x, "%"))+
               theme(panel.border = element_blank(),# panel.grid.major = element_blank(),
#panel.grid.minor = element_blank(), 
      axis.line = element_line(colour = "black"),axis.text.x = element_text(angle = 45, vjust = 1, hjust=1))+
geom_text(aes(label = round(PctPerYear,0)), hjust = 0.75,  vjust = -1) +
                   theme_minimal()+
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))

In [ ]:
library(grid)
#library(gridtext)
library(gridExtra)
plot <- list(OVERALL, gender, IMD, ethnicity)
pdf('1.2 Speech Language and Communication GLD.pdf',width=10, height=10)
  title <- "Plots for 1.2 Speech Language & Communication : Good Learning Development"
grid.text(title) 
plot
dev.off()
#